# Advanced RQ-VAE / SID-v2 PLUM-like pipeline

????: ??????????? Semantic IDs ?? ?????? MovieLens-1M movie metadata + overview ???????.

???????? ??????? ?? ??????? RQ-VAE:
- ??? ????????? ?????: `meta = title/year/genres`, `description = overview`;
- fusion MLP ????? RQ-VAE, ? ?? ????????? SID ?? ???????????;
- multi-resolution codebooks + progressive masking;
- ?????????? bidirectional InfoNCE;
- behavior positives ???????? ?? ?? ???????? ???????, ? ?? weighted PPMI co-watch graph.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.sid import (
    AdvancedRQVAETrainingConfig,
    RQVAEConfig,
    build_item_text_profiles,
    build_weighted_cooccurrence_pairs,
    encode_text_modalities,
    load_feature_bundle,
    run_advanced_rqvae_experiment,
)

ROOT

## 1. Build text profiles

?????????? ??????? `item_idx` ?? ???????, ????? ????? SID ???? ?????????? ? CPT/SFT ??????????. ???????? ????? ?? ??? ???????????? `research/movie_overviews/data/ml1m_movie_overviews.csv`.

In [ ]:
ITEM_META_PATH = ROOT / "data/processed/item_features/item_meta.parquet"
OVERVIEWS_PATH = ROOT / "research/movie_overviews/data/ml1m_movie_overviews.csv"
PROFILE_PATH = ROOT / "data/processed/item_features/rqvae_v2_BAAI_bge_large_en_v1_5_profiles.parquet"

item_meta = pd.read_parquet(ITEM_META_PATH)
overviews = pd.read_csv(OVERVIEWS_PATH)
profiles = build_item_text_profiles(item_meta, overviews)
profiles.to_parquet(PROFILE_PATH, index=False)

profiles[['item_idx', 'movie_id', 'meta_text', 'description_text']].head()

## 2. Encode meta and description

??? ???????????????? ?????????? baseline ???????????? cached `BAAI/bge-large-en-v1.5`. ????? ???? ????? ?????????? `Qwen/Qwen3-Embedding-4B`, ???? ?????? ??????? ????? ??????? embedding-??????.

In [ ]:
FEATURE_PATH = ROOT / "data/processed/item_features/rqvae_v2_BAAI_bge_large_en_v1_5_meta_desc.npz"

if FEATURE_PATH.exists():
    bundle = load_feature_bundle(FEATURE_PATH)
else:
    bundle = encode_text_modalities(
        profiles,
        output_path=FEATURE_PATH,
        model_name="BAAI/bge-large-en-v1.5",
        batch_size=64,
        device="cuda",
        local_files_only=True,
    )

bundle.n_items, bundle.modality_dims

## 3. Mine behavior positives

??? ?? adjacent-pairs. ???? ???????? ?? ???????????????? ??????? ? ?????, decay ?? ???????, ???????? `rating >= 4`, PPMI debiasing ? top-K positives ?? anchor item.

In [ ]:
TRAIN_PATH = ROOT / "data/processed/splits/train.parquet"
PAIR_PATH = ROOT / "data/processed/sid_pairs/co_pairs_behavior_ppmi_w20_r4_top32.parquet"

if PAIR_PATH.exists():
    pairs = pd.read_parquet(PAIR_PATH)
else:
    train = pd.read_parquet(TRAIN_PATH)
    pairs = build_weighted_cooccurrence_pairs(
        train,
        output_path=PAIR_PATH,
        window_size=20,
        distance_decay=0.85,
        min_rating=4.0,
        top_k_per_item=32,
    )

pairs.shape, pairs['item_idx'].nunique(), pairs['item_pos'].nunique()

In [ ]:
pairs[['co_weight', 'pmi', 'score', 'rank_for_item']].describe()

## 4. Train advanced RQ-VAE

???????? ?????? ??? ??? ?????? ?? 40 ????. ??? ????? ???????? ?????????? SID-run ????? ??????? `epochs` ?? 80-120 ? ???????? ?? `behavior_recall_at_10`, `sid_uniqueness`, `sid_entropy_norm`.

In [ ]:
OUTPUT_DIR = ROOT / "runs/advanced_rqvae_sid_v2_bge_large_ppmi"

model_config = RQVAEConfig(
    modality_dims=bundle.modality_dims,
    latent_dim=256,
    branch_dim=256,
    codebook_sizes=(2048, 1024, 512, 256),
    dropout=0.10,
)

train_config = AdvancedRQVAETrainingConfig(
    epochs=40,
    steps_per_epoch=64,
    eval_every=5,
    item_batch_size=512,
    pair_batch_size=512,
    lr=3e-4,
    contrastive_weight=0.05,
)

summary = run_advanced_rqvae_experiment(
    bundle=bundle,
    pairs=pairs,
    output_dir=OUTPUT_DIR,
    model_config=model_config,
    train_config=train_config,
)
summary

## 5. Inspect saved artifacts

In [ ]:
import json
import numpy as np

summary = json.loads((OUTPUT_DIR / "summary.json").read_text(encoding="utf-8"))
sids = np.load(OUTPUT_DIR / "SIDs.npy")

summary['final_eval'], sids.shape